## Semana 8

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler


In [ ]:
def escalar_dataset(df, test_size=0.2, random_state=42, graficar=True):
    df = df.copy()  # evita modificar el DataFrame original
    filas_entrada = len(df)
    print("Shape de entrada:", df.shape)

    # --- Rangos antes de escalar ---
    columnas = ["edad_cliente", "precio_unitario", "monto_compra", "unidades_vendidas"]
    print(df[columnas].describe())

    # --- Imputación de nulos en monto_compra (antes del split: no usa estadísticas agregadas) ---
    print("Nulos en monto_compra ANTES de imputar:", df["monto_compra"].isna().sum())
    df["monto_compra"] = df["monto_compra"].fillna(df["precio_unitario"] * df["unidades_vendidas"])
    print("Nulos en monto_compra DESPUÉS de imputar:", df["monto_compra"].isna().sum())

    # --- División entrenamiento / prueba ---
    df_entrenamiento, df_prueba = train_test_split(
        df, test_size=test_size, random_state=random_state
    )
    df_entrenamiento = df_entrenamiento.copy()
    df_prueba = df_prueba.copy()
    assert len(df_entrenamiento) + len(df_prueba) == filas_entrada

    # --- Escaladores: fit solo con entrenamiento, transform en ambos conjuntos ---
    escaladores = {
        "edad_cliente": StandardScaler(),
        "monto_compra": RobustScaler(),
        "precio_unitario": StandardScaler(),
        "unidades_vendidas": StandardScaler(),
    }
    for col, escalador in escaladores.items():
        escalador.fit(df_entrenamiento[[col]])
        df_entrenamiento[col + "_escalada"] = escalador.transform(df_entrenamiento[[col]])
        df_prueba[col + "_escalada"] = escalador.transform(df_prueba[[col]])

    escaladas = [c + "_escalada" for c in escaladores]
    print("\nColumnas escaladas (entrenamiento):")
    print(df_entrenamiento[escaladas].describe())
    print("\nColumnas escaladas (prueba):")
    print(df_prueba[escaladas].describe())

    # --- Antes y después de monto_compra, y las 4 variables en una misma escala ---
    if graficar:
        fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
        df_entrenamiento["monto_compra"].hist(ax=ejes[0])
        df_entrenamiento["monto_compra_escalada"].hist(ax=ejes[1])
        ejes[0].set_title("monto_compra (original)")
        ejes[1].set_title("monto_compra_escalada (RobustScaler)")
        plt.show()

        plt.figure(figsize=(10, 6))
        df_entrenamiento[escaladas].boxplot()
        plt.xticks(rotation=15, ha="right")
        plt.tight_layout()
        plt.show()

    # --- Unir entrenamiento y prueba en un solo df para la siguiente etapa ---
    # reset_index deja el df como si se hubiera releído el CSV de la semana 8 (índice 0..n-1)
    df = pd.concat([df_entrenamiento, df_prueba]).reset_index(drop=True)
    assert len(df) == filas_entrada
    print("\nShape de salida:", df.shape)

    return df, df_entrenamiento, df_prueba, escaladores


# Uso (df viene de tu función anterior)
df, df_entrenamiento, df_prueba, escaladores = escalar_dataset(df)

### Imputacion de nulos

## Semana 9

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from category_encoders import TargetEncoder


def codificar_dataset(df, test_size=0.2, random_state=42, umbral_postal=15,
                      ruta_salida=None):
    """
    Recibe un DataFrame (ya escalado), codifica las variables categóricas
    y divide en entrenamiento y prueba.

    - One-hot: categoria_producto, metodo_pago, canal_compra
    - Ordinal: nivel_lealtad (Bronce < Plata < Oro < Platino)
    - Target encoding: ciudad_tienda (ajustado solo con entrenamiento)
    - Agrupación de categorías raras: codigo_postal (frecuencias solo de entrenamiento)

    Retorna: df_entrenamiento, df_prueba, codificador_lealtad, codificador_ciudad
    """
    df = df.copy()  # evita modificar el DataFrame original
    print("Shape de entrada:", df.shape)

    # --- Ordinal encoding: nivel_lealtad (orden explícito, no alfabético) ---
    orden_lealtad = ["Bronce", "Plata", "Oro", "Platino"]
    codificador_lealtad = OrdinalEncoder(categories=[orden_lealtad])
    df["nivel_lealtad_cod"] = codificador_lealtad.fit_transform(df[["nivel_lealtad"]])

    # --- One-hot encoding: nominales de baja cardinalidad ---
    df_codificado = pd.get_dummies(
        df,
        columns=["categoria_producto", "metodo_pago", "canal_compra"],
        prefix=["cat", "pago", "canal"],
    )
    print("Shape después de one-hot:", df_codificado.shape)

    # --- División entrenamiento / prueba (sobre el df ya codificado) ---
    df_entrenamiento, df_prueba = train_test_split(
        df_codificado, test_size=test_size, random_state=random_state
    )
    df_entrenamiento = df_entrenamiento.copy()
    df_prueba = df_prueba.copy()
    assert len(df_entrenamiento) + len(df_prueba) == len(df_codificado)

    # --- Target encoding: ciudad_tienda (solo se ajusta con entrenamiento) ---
    codificador_ciudad = TargetEncoder(cols=["ciudad_tienda"], smoothing=10)
    codificador_ciudad.fit(df_entrenamiento[["ciudad_tienda"]],
                           df_entrenamiento["monto_compra"])

    df_entrenamiento["ciudad_tienda_cod"] = codificador_ciudad.transform(
        df_entrenamiento[["ciudad_tienda"]]
    )["ciudad_tienda"]
    df_prueba["ciudad_tienda_cod"] = codificador_ciudad.transform(
        df_prueba[["ciudad_tienda"]]
    )["ciudad_tienda"]

    # --- Agrupar códigos postales poco frecuentes ---
    # Las frecuencias se calculan solo con entrenamiento y se aplican a ambos
    conteo_postal = df_entrenamiento["codigo_postal"].astype(str).value_counts()
    postales_frecuentes = conteo_postal[conteo_postal >= umbral_postal].index

    for d in (df_entrenamiento, df_prueba):
        cp = d["codigo_postal"].astype(str)
        d["codigo_postal_agrupado"] = cp.where(cp.isin(postales_frecuentes), "Otras")

    # --- Guardado opcional ---
    if ruta_salida:
        df_entrenamiento.to_csv(ruta_salida, index=False)
        print("Guardado en:", ruta_salida)

    return df_entrenamiento, df_prueba, codificador_lealtad, codificador_ciudad


# Uso (df viene de tu función anterior)
df_entrenamiento, df_prueba, codificador_lealtad, codificador_ciudad = codificar_dataset(
    df, ruta_salida="S09_PD_Diaz_DatasetCodificado.csv"
)

## Semana 10

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer


def transformar_dataset(df, test_size=0.2, random_state=42):
    """
    Recibe un DataFrame ya codificado, crea variables temporales, de texto
    y de edad, aplica transformaciones al monto y divide en entrenamiento
    y prueba.

    Retorna: df (con las nuevas columnas), df_entrenamiento, df_prueba, transformador
    """
    df = df.copy()  # evita modificar el DataFrame original
    print("Dimensiones:", df.shape)

    # --- Variables temporales ---
    df["fecha_pedido"] = pd.to_datetime(df["fecha_pedido"])
    df["mes_pedido"] = df["fecha_pedido"].dt.month
    df["trimestre_pedido"] = df["fecha_pedido"].dt.quarter
    df["dia_semana_pedido"] = df["fecha_pedido"].dt.day_name()
    df["es_fin_semana_pedido"] = df["fecha_pedido"].dt.dayofweek >= 5

    # --- Variables de texto ---
    df["comentario_cliente"] = df["comentario_cliente"].fillna("")
    df["longitud_comentario"] = df["comentario_cliente"].str.len()
    df["num_palabras_comentario"] = df["comentario_cliente"].str.split().str.len()
    df["tiene_comentario"] = df["longitud_comentario"] > 0
    df["menciona_demora"] = df["comentario_cliente"].str.contains(
        "tarde|demora|retraso|no llegó", case=False, na=False
    )

    # --- Grupos de edad ---
    df["grupo_edad"] = pd.cut(
        df["edad_cliente"],
        bins=[17, 25, 35, 45, 60, 100],
        labels=["18 a 25", "26 a 35", "36 a 45", "46 a 60", "61 y más"],
    )
    if df["grupo_edad"].isna().any():
        print("Aviso: hay edades fuera del rango 18-100 (quedaron como NaN).")

    # --- Transformación logarítmica ---
    print("Asimetría de monto_compra:", df["monto_compra"].skew())
    df["monto_compra_log"] = np.log1p(df["monto_compra"])

    # --- División entrenamiento / prueba ---
    df_entrenamiento, df_prueba = train_test_split(
        df, test_size=test_size, random_state=random_state
    )
    df_entrenamiento = df_entrenamiento.copy()
    df_prueba = df_prueba.copy()

    # --- Transformación de potencia (ajustada solo con entrenamiento) ---
    metodo = "box-cox" if (df_entrenamiento["monto_compra"] > 0).all() else "yeo-johnson"
    print("Método de transformación de potencia:", metodo)

    transformador = PowerTransformer(method=metodo)
    transformador.fit(df_entrenamiento[["monto_compra"]])

    df_entrenamiento["monto_compra_boxcox"] = transformador.transform(
        df_entrenamiento[["monto_compra"]]
    )
    df_prueba["monto_compra_boxcox"] = transformador.transform(
        df_prueba[["monto_compra"]]
    )

    return df, df_entrenamiento, df_prueba, transformador


# Uso (df viene de tu función anterior)
df, df_entrenamiento, df_prueba, transformador = transformar_dataset(df)

## Diccionario de datos de las semanas 8, 9 y 10



| Columna del dataset | De qué semana viene | Qué significa |
|---|---|---|
| `edad_cliente_escalada` | Semana 8 | `edad_cliente` estandarizada con Z-score (`StandardScaler` ajustado solo con entrenamiento). Se conserva junto a la original |
| `monto_compra_escalada` | Semana 8 | `monto_compra` con escalado robusto (`RobustScaler`: mediana e IQR de entrenamiento), para que la cola de ventas corporativas no distorsione la escala |
| `precio_unitario_escalada` | Semana 8 | `precio_unitario` estandarizada con Z-score (`StandardScaler` ajustado solo con entrenamiento) |
| `unidades_vendidas_escalada` | Semana 8 | `unidades_vendidas` estandarizada con Z-score (`StandardScaler` ajustado solo con entrenamiento) |
| `cat_<categoría>` | Semana 9 | Variables dummy (one-hot) de `categoria_producto`: una columna por categoría (6). Vale 1 si el pedido es de esa categoría y 0 si no |
| `pago_<método>` | Semana 9 | Variables dummy (one-hot) de `metodo_pago`: una columna por método de pago (4) |
| `canal_<canal>` | Semana 9 | Variables dummy (one-hot) de `canal_compra`: una columna por canal (4) |
| `nivel_lealtad_cod` | Semana 9 | Nivel de lealtad codificado en su orden real: Bronce=0, Plata=1, Oro=2, Platino=3 |
| `ciudad_tienda_cod` | Semana 9 | Target encoding de `ciudad_tienda`: promedio suavizado (smoothing=10) de `monto_compra` por ciudad, calculado solo con entrenamiento |
| `codigo_postal_agrupado` | Semana 9 | `codigo_postal` donde los códigos con menos de 15 pedidos en entrenamiento se agrupan como "Otras" |
| `mes_pedido` | Semana 10 | Mes en que se hizo el pedido (1 a 12), extraído de `fecha_pedido` |
| `trimestre_pedido` | Semana 10 | Trimestre del año del pedido (1 a 4), extraído de `fecha_pedido` |
| `dia_semana_pedido` | Semana 10 | Nombre del día de la semana del pedido (Monday, Tuesday, ...) |
| `es_fin_semana_pedido` | Semana 10 | True si el pedido se hizo en sábado o domingo, False en caso contrario |
| `longitud_comentario` | Semana 10 | Número de caracteres de `comentario_cliente` (0 si no hay comentario) |
| `num_palabras_comentario` | Semana 10 | Número de palabras de `comentario_cliente` |
| `tiene_comentario` | Semana 10 | True si el cliente dejó un comentario, False si el campo estaba vacío |
| `menciona_demora` | Semana 10 | True si el comentario contiene "tarde", "demora", "retraso" o "no llegó" |
| `grupo_edad` | Semana 10 | Rango de edad del cliente: 18 a 25, 26 a 35, 36 a 45, 46 a 60, 61 y más |
| `monto_compra_log` | Semana 10 | Logaritmo natural de (1 + `monto_compra`), para reducir su sesgo |
| `monto_compra_boxcox` | Semana 10 | `monto_compra` transformado con PowerTransformer (Box-Cox, o Yeo-Johnson si hay valores ≤ 0), ajustado solo con entrenamiento. Solo existe en los conjuntos de entrenamiento y prueba |